In [ ]:
import gc
import torch

# 先清理一下上一个 notebook 可能残留的显存缓存。
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print('GPU cache cleared.')
else:
    print('CUDA not available, skip GPU cache clear.')


# 下一课：连续动作版策略梯度入门

前一课我们已经理解了连续动作空间最核心的概念：
- 动作不再是几个离散类别
- 策略网络通常输出 `mean` 和 `std`
- 然后用 `Normal` 分布采样动作

这一课我们把这件事真正用在训练里，做一个连续动作版的策略梯度入门 demo。

重点不是追求最强性能，而是让你第一次真正看到：

**连续动作策略是怎么参与训练的。**


## 1. 为什么连续动作会更麻烦

在 `CartPole` 这种离散动作环境里，我们可以直接：
- 输出每个动作的 logits
- 用 `Categorical` 采样动作

但在 `Pendulum` 这种连续动作环境里，我们必须面对两个新问题：

1. 动作是连续值，不是有限类别
2. 动作通常还有上下界，不能无限大

所以这节课里，你会看到：
- 用 `Normal(mean, std)` 表示策略
- 用 `tanh` 把动作压回合理范围


In [ ]:
import random
import warnings

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=3, suppress=True)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


In [ ]:
def pick_device():
    if torch.cuda.is_available():
        try:
            _ = torch.zeros(1, device='cuda')
            return torch.device('cuda')
        except Exception as e:
            warnings.warn(f'检测到 CUDA，但当前环境无法真正使用 GPU，已回退到 CPU。原因: {e}')
    return torch.device('cpu')


device = pick_device()
print('当前设备:', device)
if torch.cuda.is_available():
    print('检测到 CUDA 设备:', torch.cuda.get_device_name(0))


In [ ]:
env = gym.make('Pendulum-v1')
state, info = env.reset(seed=42)

print('初始状态:', state)
print('状态维度:', env.observation_space.shape[0])
print('动作维度:', env.action_space.shape[0])
print('动作下界:', env.action_space.low)
print('动作上界:', env.action_space.high)


## 2. 连续动作策略网络

这里我们让网络输出：
- `mean`
- `log_std`

然后：
- `std = exp(log_std)`
- 用 `Normal(mean, std)` 构造策略分布

最后从这个分布中采样动作。


In [ ]:
def to_tensor(x, device):
    return torch.tensor(x, dtype=torch.float32, device=device)


class ContinuousPolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.mean_head = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))

    def forward(self, x):
        features = self.net(x)
        mean = self.mean_head(features)
        log_std = self.log_std.expand_as(mean)
        std = torch.exp(log_std)
        return mean, std


## 3. 为什么要做 `tanh` 和动作缩放

`Normal` 分布采样出来的动作理论上可以无限大，
但环境动作范围通常是有限的，比如 `Pendulum` 里是 `[-2, 2]`。

常见做法是：
- 先采样一个 `raw_action`
- 再过 `tanh` 压到 `[-1, 1]`
- 再映射到环境动作范围

这一步在连续动作算法里非常常见。


In [ ]:
action_scale = torch.tensor((env.action_space.high - env.action_space.low) / 2.0, dtype=torch.float32, device=device)
action_bias = torch.tensor((env.action_space.high + env.action_space.low) / 2.0, dtype=torch.float32, device=device)

def sample_action_and_log_prob(policy_net, state_tensor):
    mean, std = policy_net(state_tensor)
    dist = Normal(mean, std)

    raw_action = dist.rsample()
    squashed_action = torch.tanh(raw_action)
    action = squashed_action * action_scale + action_bias

    # 教学简化版：这里只保留高斯分布下的 log_prob 主线，
    # 不深入展开更严格的 tanh 修正项推导。
    log_prob = dist.log_prob(raw_action).sum(dim=1)
    return action, log_prob, mean, std


## 4. 连续动作版 REINFORCE 主线

这节课我们继续沿用最容易理解的 REINFORCE 形式：
- 收集一整条轨迹
- 计算每一步 return
- 用 `-log_prob * return` 更新策略

和离散动作版相比，主线并没有变。
变的只是：
- 动作分布从 `Categorical` 变成了 `Normal`
- 动作需要做连续值处理和范围映射


In [ ]:
def compute_returns(rewards, gamma=0.99):
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    return returns


In [ ]:
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

policy_net = ContinuousPolicyNet(state_dim, action_dim, hidden_dim=128).to(device)
optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)

gamma = 0.99
episodes = 120
max_steps = 200

episode_rewards = []
loss_history = []
mean_history = []
std_history = []

for episode in range(episodes):
    state, info = env.reset()
    log_probs = []
    rewards = []
    total_reward = 0.0

    for step in range(max_steps):
        state_tensor = to_tensor(state, device).unsqueeze(0)
        action_tensor, log_prob, mean, std = sample_action_and_log_prob(policy_net, state_tensor)

        action = action_tensor.squeeze(0).detach().cpu().numpy()
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        log_probs.append(log_prob.squeeze(0))
        rewards.append(reward)
        total_reward += reward
        state = next_state

        if done:
            break

    returns = compute_returns(rewards, gamma=gamma).to(device)

    policy_loss = []
    for log_prob, G in zip(log_probs, returns):
        policy_loss.append(-log_prob * G)
    loss = torch.stack(policy_loss).sum()

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy_net.parameters(), max_norm=10.0)
    optimizer.step()

    episode_rewards.append(total_reward)
    loss_history.append(float(loss.item()))
    mean_history.append(float(mean.mean().item()))
    std_history.append(float(std.mean().item()))

print('训练完成。')
print('最后 20 轮平均 reward:', round(float(np.mean(episode_rewards[-20:])), 2))


## 5. 看训练曲线

这里我们除了看 reward 和 loss，还顺手看一下：
- `mean` 的平均变化
- `std` 的平均变化

这样你能更直观看到连续动作策略分布是怎么变的。


In [ ]:
window = 10
smoothed_rewards = np.convolve(episode_rewards, np.ones(window) / window, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(episode_rewards, color='#1f77b4', alpha=0.5, label='raw')
axes[0, 0].plot(range(window - 1, len(episode_rewards)), smoothed_rewards, color='#d62728', label='smoothed')
axes[0, 0].set_title('每轮 reward')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Reward')
axes[0, 0].legend()

axes[0, 1].plot(loss_history, color='#2ca02c', alpha=0.7)
axes[0, 1].set_title('训练损失')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Loss')

axes[1, 0].plot(mean_history, color='#ff7f0e', alpha=0.7)
axes[1, 0].set_title('Mean 平均变化')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Mean')

axes[1, 1].plot(std_history, color='#9467bd', alpha=0.7)
axes[1, 1].set_title('Std 平均变化')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Std')

plt.tight_layout()
plt.show()


## 6. 用当前策略跑一轮看看

这里我们让训练后的策略跑一轮，观察最终 reward。

连续动作环境通常比 `CartPole` 更难，所以不要期待一开始就有特别漂亮的分数。
这节课的主要目的仍然是理解训练主线，而不是冲最优性能。


In [ ]:
test_env = gym.make('Pendulum-v1')
state, info = test_env.reset(seed=123)
test_reward = 0.0

policy_net.eval()
with torch.no_grad():
    for step in range(200):
        state_tensor = to_tensor(state, device).unsqueeze(0)
        mean, std = policy_net(state_tensor)

        # 测试时直接用均值动作，减少随机性
        action_tensor = torch.tanh(mean) * action_scale + action_bias
        action = action_tensor.squeeze(0).cpu().numpy()

        state, reward, terminated, truncated, info = test_env.step(action)
        test_reward += reward
        if terminated or truncated:
            break

print('测试轮 reward:', round(float(test_reward), 2))
test_env.close()


## 7. 这节课最该记住什么

如果你想抓住这节课的核心变化，就记住：

- 离散动作策略输出类别分布
- 连续动作策略输出高斯分布参数
- 训练时仍然可以沿用策略梯度主线：`-log_prob * return`

所以策略梯度的思想没变，变的是动作分布的形式。


## 8. 下一课最自然学什么

学完这一课后，下一步最自然的方向有两个：
- 连续动作版本的 `Actor-Critic / PPO`
- 或直接进入 `DDPG / TD3 / SAC`

如果按学习路线来说，我建议下一课先做一个“连续动作版 Actor-Critic / PPO 过渡课”。
